In [1]:
"""
02_dataset_cleanup.py

Prepare the raw dataset for further analysis:
- drop unused/duplicate columns
- convert numeric and date columns to proper dtypes
- drop columns that are almost entirely missing
- save the cleaned dataset + a log of what was removed
"""


'\n02_dataset_cleanup.py\n\nPrepare the raw dataset for further analysis:\n- drop unused/duplicate columns\n- convert numeric and date columns to proper dtypes\n- drop columns that are almost entirely missing\n- save the cleaned dataset + a log of what was removed\n'

In [2]:
from config import DATASET_CLEANED, LOGS, MISSING_THRESHOLD
from utils.load_data_raw import load_data_raw
from utils.convert_comma_to_decimal import convert_comma_to_decimal
from utils.parse_yyyymmdd_to_date import parse_yyyymmdd
from utils.parse_sas_date import parse_sas_date
from utils.print_section import print_section

# -------------------------------------------------------------------
# Columns from the raw dataset that should be deleted
# -------------------------------------------------------------------
COLUMNS_TO_DELETE = [
    'score', 'ratio12', 'ratio14', 'ratio17', 'ratio18', 'ratio19',
    'score_w', 'z_score_rel', 'z_score_tot', 'z_score_rel_w',
    'z_score_tot_w', 'z_addscore', 'z_add',
    'dgrmovescount', 'dgractionpubs', 'dgrstreetfails',
    'dgrstreetstop', 'dgrstreetall', 'dgrpersred',
    'dgrpersgr', 'dgrpersyel', 'dgrpersblue',
    'dgrstreet_ratio', 'total_actions', 'action_rate',
    'afsluit', 'neerlegging', 'obs', 'obser', 'append', 'tag',
    'q_laat_ratio', 'delta_aantal_dagen', 'delta_ad_abs',
    'sum_delta', 'avg_delta', 'aantal_dagen_w',
    'delta_aantal_dagen_w', 'som_laat', 'som_laat_drie',
    'laat_ratio', 'laatstejaar_laat',
    'laatstejaar_laat_dummy', 'avg_delta_w', 'q_avg_delta',
    'avg_delta_pre', 'laat_ratio_dummy', 'laat_dummy_pre',
    'laat_ratio_pre', 'laat_ord', 'fail_dummy_drie',
    'trade_debt', 'fin_debt', 'total_liabilities',
    'lev', 'current_ratio', 'current_ratio_pre',
    'current_dummy', 'net_income_pre', 'loss_dummy',
    'growth', 'lev_w', 'size_w', 'age_w',
    'growth_w', 'trade_debt_w', 'fin_debt_w',
    'move_rate_w', 'fail_afstand',
    'laatste_jaar', 'jvo'
]

# ------------------------------------------------------------------
# Columns that are duplicates of anothers under a different name
# ------------------------------------------------------------------
DUPLICATE_COLUMNS = [
    'Rub16', 'rub16', 'rub29_58', 'rub170_4', 'rub43', 'rub21',
    'rub9904', 'rub42_48', 'rub20_58', 'rub17_49', 'rub175', 'rub44',
    'nace',
]

# -----------------------------------------------------------------
# Date columns stored as YYYYMMDD integers vs. SAS day-offset floats.
# ------------------------------------------------------------------
DATE_COLUMNS_YYYYMMDD = [
    'startdate', 'closedate', 'deposit', 'oprichting',
    'stopdatum', 'nglnbb', 'fail',
]
DATE_COLUMNS_SAS = ['faling_datum']

# -----------------------------------------------------------------
# Numeric columns that should not be converted 
# -----------------------------------------------------------------

EXCLUDE_COLUMNS = [
    'vat', 'bookyear', 'nature', 'hoofdnacebel', 'industry',
    'rechtsvorm', 'stopreden', 'jaar_van_faling', 'fail_dummy',
]

# -----------------------------------------------------------------
# Target-related columns that must survive the missing-value 
# (e.g. faling_datum is empty for healthy firms).
# -----------------------------------------------------------------

PROTECTED_COLUMNS = [
    'jaar_van_faling', 'fail', 'faling_datum', 'fail_dummy',
]

# ------------------------------------------------------------------
# Load the dataset
# ------------------------------------------------------------------
print_section("Load raw data")

df = load_data_raw()
# Save columns and rows for the log
original_rows, original_columns = df.shape
print(df.shape)

# ------------------------------------------------------------------
# Remove unused and duplicate columns
# ------------------------------------------------------------------
print_section("Remove unused variables")

dropped_unused_columns = [c for c in COLUMNS_TO_DELETE if c in df.columns]
df = df.drop(columns=dropped_unused_columns)
print(f"Dropped {len(dropped_unused_columns)} columns")

print_section("Remove duplicate variables")

dropped_duplicate_columns = [c for c in DUPLICATE_COLUMNS if c in df.columns]
df = df.drop(columns=dropped_duplicate_columns)
print(f"Dropped {len(dropped_duplicate_columns)} duplicate columns")

# ------------------------------------------------------------------
# Convert numeric / date columns
# ------------------------------------------------------------------
print_section("Convert numeric columns")

numeric_cols = [
    c for c in df.columns
    if c not in EXCLUDE_COLUMNS + DATE_COLUMNS_YYYYMMDD + DATE_COLUMNS_SAS
]
for col in numeric_cols:
    df[col] = convert_comma_to_decimal(df[col])
print(f"Converted {len(numeric_cols)} columns")

print_section("Convert date columns")

for col in DATE_COLUMNS_YYYYMMDD:
    if col in df.columns:
        df[col] = parse_yyyymmdd(df[col])
for col in DATE_COLUMNS_SAS:
    if col in df.columns:
        df[col] = parse_sas_date(df[col])
print("Date conversion complete")

# ------------------------------------------------------------------
# Drop columns that are almost entirely missing
# ------------------------------------------------------------------
print_section("Remove highly missing columns")

missing_pct = df.isna().mean()
dropped_missing_columns = [
    col for col in missing_pct[missing_pct >= MISSING_THRESHOLD].index
    if col not in PROTECTED_COLUMNS
]
df = df.drop(columns=dropped_missing_columns)
print(f"Dropped {len(dropped_missing_columns)} columns with >= {MISSING_THRESHOLD:.0%} missing values")

# ------------------------------------------------------------------
# Save the cleaned dataset
# ------------------------------------------------------------------
print_section("Final dataset")
print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns):,}")

print_section("Save cleaned data")
df.to_csv(DATASET_CLEANED, index=False)
print(DATASET_CLEANED)

# ------------------------------------------------------------------
# Log what columns were dropped
# ------------------------------------------------------------------
print_section("Write drop log")

LOG_PATH = LOGS / "data_cleanup_log.txt"

with open(LOG_PATH, "w") as f:
    f.write("DATA CLEANING LOG\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Original dataset: {original_rows:,} rows x {original_columns:,} columns\n\n")
    f.write(f"Final dataset: {len(df):,} rows x {len(df.columns):,} columns\n\n")
    f.write(f"MISSING THRESHOLD: {MISSING_THRESHOLD:.0%}\n\n")

    for title, cols in [
        ("UNUSED COLUMNS REMOVED", dropped_unused_columns),
        ("DUPLICATE COLUMNS REMOVED", dropped_duplicate_columns),
        ("HIGH-MISSING COLUMNS REMOVED", dropped_missing_columns),
    ]:
        f.write(f"{title} ({len(cols)})\n")
        f.write("-" * 60 + "\n")
        for col in sorted(cols):
            f.write(f"{col}\n")
        f.write("\n\n")

print(f"Log written to: {LOG_PATH}")



Load raw data


(4502867, 286)

Remove unused variables
Dropped 71 columns

Remove duplicate variables


Dropped 13 duplicate columns

Convert numeric columns


Converted 185 columns

Convert date columns


Date conversion complete

Remove highly missing columns


Dropped 96 columns with >= 90% missing values

Final dataset
Rows    : 4,502,867
Columns : 106

Save cleaned data


/Users/seymaciftci/Code/bankruptcy-master-thesis/data/processed/dataset_cleaned.csv

Write drop log
Log written to: /Users/seymaciftci/Code/bankruptcy-master-thesis/logs/data_cleanup_log.txt
